## Si consideri il data set CIFAR10 che contiene un training set di 50.000 immagini a colori aventi dimensione 32x32 e articolate in 10 categorie diverse. Il test set è costituito da 10.000 immagini di test con le stesse dimensioni e le stesse etichette di classe.

### 1. Caricare il data set tramite la Torch ed espungere randomicamente un validation set pari al 10% dei dati di training, mantenendo il restante 90% come training set vero e proprio. Effettuare una data augmentatioin usando mirroring orizzontali e verticali casuali nonché traslazioni e rotazioni casuali in un range di +-5 pixel in x e y +-10 rispettivamente.

In [1]:
# Importiamo pytorch
import torch
from torchvision import datasets, transforms
from torch.utils.data import Subset, random_split
from torchnn import make_dataloaders


# Controlla se i driver CUDA (per GPU NVIDIA) sono disponibili
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Il codice verrà eseguito su: {device}")
# Ricordiamo che una data augmentation è una tecnica fondamentale nel ML utilizzata per espandere artificialmente 
# le dimensioni e la diversità di un dataset di addestramento, creando versioni modificate dei dati originali.
# Si utilizza per migliorare le prestazioni e la robustezza dei modelli predittivi risolvendo problemi che si incontrano nella fase
# di training come l'overfitting, la non generalizzazione, i costi di raccolta e lo sbilanciamento delle classi.

#Tensore


# DATA AUGMENTATION
#            = transforms.Compose([Mirroring orizzontale randomize, Mirroring verticale randomize, .RandomAffine(degrees=10, translate=(traslazione/dim.immagine, traslazione/dim.immagine)),
#                                  Converti in tensore])
tr_transform = transforms.Compose([transforms.RandomHorizontalFlip(),#Mirroring casuale
                                   transforms.RandomVerticalFlip(),  #Mirroring casuale
                                   transforms.RandomAffine(degrees=10, translate=(5/32, 5/32,)), #Trasformazione affine del tipo: rotazione, traslo+, traslo-
                                   transforms.ToTensor() #Tensore Pytorch
                                   ])

te_transform = transforms.Compose([transforms.ToTensor()])

# Carico il dataset

data_path = './cifar-10-python'

# training
tr_dataset = datasets.CIFAR10(root = data_path, train = True, download = False, transform = tr_transform)
# validation
val_dataset = datasets.CIFAR10(root = data_path, train = True, download = False, transform = te_transform)
# test
te_dataset = datasets.CIFAR10(root = data_path, train = False, download = False, transform = te_transform)

# SPLIT del 10% per val set
n_train = len(tr_dataset)
tr_size = int(0.9 * n_train) 
val_size = n_train - tr_size 

# Indici casuali
train_subset_temp, val_subset_temp = random_split(tr_dataset, [tr_size, val_size])
# Creazione Subset incrociati con le trasformazioni corrette
train_data = Subset(tr_dataset, train_subset_temp.indices)
val_data = Subset(val_dataset, val_subset_temp.indices)

# DATA LOADERS 
train_dl, val_dl, test_dl = make_dataloaders(train_data, val_data, te_dataset)

Il codice verrà eseguito su: cuda
Shape e tipo dei campioni: torch.Size([64, 3, 32, 32]), torch.float32
Shape e tipo delle etichette: torch.Size([64]) torch.int64


### 2. Implementare un classificatore dei punti del data set processato al punto 1, utilizzando un algoritmo di Gradient Boosting con la seguente griglia di ricerca, usando una cross-validation a 10 fold:
### - Learning rate: 0.01, 0.1, 0.2
### - N.stimatori: 50, 100, 200
### - Min. campioni per foglia: 10, 20 , 50
### Stampare l'accuracy del miglior modello in addestramento e predizione


In [ ]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier 
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
import time

print("1. Estrazione e conversione delle immagini in vettori per Scikit-Learn...")
# Creiamo le liste vuote per X_train e y_train
X_train_list = []
y_train_list = []

# Estraiamo le immagini dal dataset di training (Punto 1)
# flatten() trasforma l'immagine 3x32x32 in un singolo vettore di 3072 numeri
for img, label in train_data:
    X_train_list.append(img.numpy().flatten())
    y_train_list.append(label)

X_train = np.array(X_train_list)
y_train = np.array(y_train_list)
print(f"   Shape di X_train: {X_train.shape}")

# ---------------------------------------------------------

print("\n2. Configurazione GridSearchCV (Stile Es5)...")

# Iperparametri richiesti dalla consegna
iperparametri = {
    'learning_rate': [0.01, 0.1, 0.2],
    'max_iter': [50, 100, 200],         # max_iter è l'equivalente di n_estimators qui
    'min_samples_leaf': [10, 20, 50]
}

# Inizializziamo il classificatore
classificatore = HistGradientBoostingClassifier(random_state=42)

# GridSearch con cross-validation a 10 fold come richiesto
grid_search = GridSearchCV(
    estimator = classificatore,
    param_grid = iperparametri,
    cv = 10,
    scoring = 'accuracy',
    n_jobs = -1,
    verbose = 2 # Stampa a video l'avanzamento dei calcoli
)

# ---------------------------------------------------------

print("\n3. Inizio la ricerca degli iperparametri migliori con GridSearchCV...")
start_time = time.time()

# Eseguo l'addestramento sui dati di training estratti
grid_search.fit(X_train, y_train)

print(f"\nRicerca completata in {(time.time() - start_time)/60:.2f} minuti.")

# Visualizziamo i risultati
print("\n--- Risultati Ottimizzazione ---")
print(f"Migliori iperparametri trovati: {grid_search.best_params_}")
print(f"Miglior accuratezza in Cross-Validation: {grid_search.best_score_:.4f}")

# Estraiamo il modello "vincitore"
best_gb_model = grid_search.best_estimator_

# ---------------------------------------------------------

print("\n4. Calcolo Accuracy in addestramento e predizione (Validation)...")

# Dobbiamo estrarre anche il Validation Set per testare la "predizione"
X_val_list = []
y_val_list = []
for img, label in val_data:
    X_val_list.append(img.numpy().flatten())
    y_val_list.append(label)

X_val = np.array(X_val_list)
y_val = np.array(y_val_list)

# Predizioni finali
y_train_pred = best_gb_model.predict(X_train)
y_val_pred = best_gb_model.predict(X_val)

print(f"Accuracy in Addestramento (Train): {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Accuracy in Predizione (Validation): {accuracy_score(y_val, y_val_pred):.4f}")

1. Estrazione e conversione delle immagini in vettori per Scikit-Learn...
   Shape di X_train: (45000, 3072)

2. Configurazione GridSearchCV (Stile Es5)...

3. Inizio la ricerca degli iperparametri migliori con GridSearchCV...
Fitting 10 folds for each of 27 candidates, totalling 270 fits


KeyboardInterrupt: 

In [3]:
import numpy as np
import time
import warnings

# Questo import speciale serve perché l'Halving è una feature avanzata di sklearn
from sklearn.experimental import enable_halving_search_cv 
from sklearn.model_selection import HalvingGridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore')

print("1. Estrazione e conversione delle immagini in vettori...")
# (Assicurati di aver applicato la scala di grigi nel Punto 1!)
X_train_list = []
y_train_list = []

for img, label in train_data:
    X_train_list.append(img.numpy().flatten())
    y_train_list.append(label)

X_train = np.array(X_train_list)
y_train = np.array(y_train_list)

# ---------------------------------------------------------

print("\n2. Configurazione HalvingGridSearchCV (Ricerca super-efficiente)...")

iperparametri = {
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [50, 100, 200],         
    'min_child_weight': [10, 20, 50] 
}

# XGBoost impostato su GPU
classificatore = XGBClassifier(
    random_state=42,
    tree_method='hist',
    device='cuda',
)

# Il Torneo ad eliminazione!
halving_search = HalvingGridSearchCV(
    estimator = classificatore,
    param_grid = iperparametri,
    cv = 5,           # Abbassato a 5 fold per velocizzare ulteriormente (10 è davvero eccessivo)
    factor = 3,       # A ogni round, tiene solo 1/3 dei candidati migliori
    scoring = 'accuracy',
    verbose = 1
)

# ---------------------------------------------------------

print("\n3. Inizio il torneo degli iperparametri...")
start_time = time.time()

halving_search.fit(X_train, y_train)

print(f"\nRicerca completata in {(time.time() - start_time)/60:.2f} minuti.")

# Estraiamo il modello "vincitore"
best_gb_model = halving_search.best_estimator_

# ---------------------------------------------------------

print("\n4. Calcolo Accuracy in addestramento e predizione (Validation)...")

X_val_list = []
y_val_list = []
for img, label in val_data:
    X_val_list.append(img.numpy().flatten())
    y_val_list.append(label)

X_val = np.array(X_val_list)
y_val = np.array(y_val_list)

y_train_pred = best_gb_model.predict(X_train)
y_val_pred = best_gb_model.predict(X_val)

print(f"Accuracy in Addestramento (Train): {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Accuracy in Predizione (Validation): {accuracy_score(y_val, y_val_pred):.4f}")

1. Estrazione e conversione delle immagini in vettori...

2. Configurazione HalvingGridSearchCV (Ricerca super-efficiente)...

3. Inizio il torneo degli iperparametri...
n_iterations: 4
n_required_iterations: 4
n_possible_iterations: 4
min_resources_: 1666
max_resources_: 45000
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 27
n_resources: 1666
Fitting 5 folds for each of 27 candidates, totalling 135 fits


KeyboardInterrupt: 

### 3. Implementare una piccola rete neurale convoluzionale bidimensionale in
### PyTorch, utilizzando la semplice API torchnn.py, per eseguire la
### classificazione multiclasse. La rete dovrà avere un adeguato numero di layer
### convoluzionali bidimensionali e, nella parte densa, dropout 0.2. Si utilizzi per
### l’addestramento l’ottimizzatore SGD con momento di Nesterov e learning
### rate dercescente esponenzialmente a partire da 0.01. Implementare una
### callback di early stopping con una pazienza sulla validation loss di 5 epoche e
### un incremento minimo di miglioramento pari a 0.01; implementare anche una
### callback di model checkpoint per il salvataggio del solo miglior modello
### rispetto alla minima validation loss.

### 4. Conservare la lista delle accuracy di addestramento e di test su tutte le epoche dell classificatore neurale e stamparne il grafico. Confrontare i risultati del miglior classificatore Gradient Boosting e del classificatore neurale calcolando e stampando, per ciascuno, la matrice di confusione, il valore di accuracy e di loss, la ROC e il valore AUC calcolati sul test set in modalità 'one-vs-rest' e media 'macro'.